<a href="https://colab.research.google.com/github/olumayowaadeniyi68-arch/AVCAD-EXERCISE/blob/main/Exercise5_Hypothesis_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [31]:
# Load dataset

df = pd.read_csv(
    "EFIplus_medit.csv",
    sep=';'
)

df.head()

,Site_code,Latitude,Longitude,Country,Catchment_name,Galiza,Subsample,Calib_EFI_Medit,Calib_connect,Calib_hydrol,...,Squalius malacitanus,Squalius pyrenaicus,Squalius torgalensis,Thymallus thymallus,Tinca tinca,Zingel asper,Squalius sp,Barbatula sp,Phoxinus sp,Iberochondrostoma_sp
0,ES_01_0002,38.102003,-4.096070,Spain,Guadalquivir,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,ES_02_0001,40.530188,-1.887796,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,ES_02_0002,40.595432,-1.928079,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,ES_02_0003,40.656184,-1.989831,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,ES_02_0004,40.676402,-2.036274,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [32]:
print(df.columns.tolist())

['Site_code', 'Latitude', 'Longitude', 'Country', 'Catchment_name', 'Galiza', 'Subsample', 'Calib_EFI_Medit', 'Calib_connect', 'Calib_hydrol', 'Calib_morphol', 'Calib_wqual', 'Geomorph1', 'Geomorph2', 'Geomorph3', 'Water_source_type', 'Flow_regime', 'Altitude', 'Geological_typology', 'Actual_river_slope', 'Natural_sediment', 'Elevation_mean_catch', 'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul', 'Barriers_catchment_down', 'Barriers_river_segment_up', 'Barriers_river_segment_down', 'Barriers_number_river_segment_up', 'Barriers_number_river_segment_down', 'Barriers_distance_river_segment_up', 'Barriers_distance_river_segment_down', 'Impoundment', 'Hydropeaking', 'Water_abstraction', 'Hydro_mod', 'Temperature_impact', 'Velocity_increase', 'Reservoir_flushing', 'Sedimentation', 'Channelisation', 'Cross_sec', 'Instream_habitat', 'Riparian_vegetation', 'Embankment', 'Floodprotection', 'Floodplain', 'Toxic_substances', 'Acidification', 'Water_quality_index', 'Eutrophication', 'Organic_p

In [33]:
# Trout present
present = df[
    df['Salmo trutta fario'] == 1
]['temp_ann']

# Trout absent
absent = df[
    df['Salmo trutta fario'] == 0
]['temp_ann']

print(present.head())
print(absent.head())

1     9.3
2    10.1
3    10.1
4    10.3
5    10.6
Name: temp_ann, dtype: float64
0     17.6
26    12.5
27    12.6
28    12.6
29    12.5
Name: temp_ann, dtype: float64


In [34]:
# Independent t-test

t_stat, p_value = stats.ttest_ind(
    present.dropna(),
    absent.dropna()
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -43.45438419732505
P-value: 0.0


# Exercise 1 Interpretation

Null hypothesis (H0):
The mean annual temperature is equal between trout presence and absence sites.

Alternative hypothesis (H1):
The mean annual temperature differs between trout presence and absence sites.

If p-value < 0.05:
- reject H0
- conclude temperature differs significantly between groups

If p-value ≥ 0.05:
- fail to reject H0

In [35]:
# Standardize temperature

df['temp_ann_std'] = stats.zscore(
    df['temp_ann'],
    nan_policy='omit'
)

df[['temp_ann', 'temp_ann_std']].head()

,temp_ann,temp_ann_std
0,17.6,1.997177
1,9.3,-1.824026
2,10.1,-1.455717
3,10.1,-1.455717
4,10.3,-1.363640


In [36]:
# Standardized temperature groups

present_std = df[
    df['Salmo trutta fario'] == 1
]['temp_ann_std']

absent_std = df[
    df['Salmo trutta fario'] == 0
]['temp_ann_std']

# T-test
t_stat, p_value = stats.ttest_ind(
    present_std.dropna(),
    absent_std.dropna()
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -43.45438419732502
P-value: 0.0


In [37]:
# Contingency table

contingency_table = pd.crosstab(
    df['Country'],
    df['Salmo trutta fario']
)

print(contingency_table)

Salmo trutta fario     0     1
Country                       
France                13    59
Italy                109    76
Portugal             615   252
Spain               1239  2648


In [38]:
# Chi-square test

chi2, p, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2)
print("P-value:", p)

Chi-square statistic: 496.3723854072799
P-value: 2.9162328651936495e-107


# Exercise 2 Interpretation

Null hypothesis (H0):
Trout presence and country are independent.

Alternative hypothesis (H1):
Trout presence and country are associated.

If p-value < 0.05:
- reject H0
- conclude trout presence depends on country

If p-value ≥ 0.05:
- fail to reject H0

In [39]:
# Top 8 catchments

top8 = df['Catchment_name'].value_counts().head(8)

print(top8)

Catchment_name
Ebro            736
Galiza-Norte    709
Minho           707
Tejo            509
Cantabrica      502
Douro           401
Guadia          313
Catala          242
Name: count, dtype: int64


In [40]:
top8_names = top8.index

print(top8_names)

Index(['Ebro', 'Galiza-Norte', 'Minho', 'Tejo', 'Cantabrica', 'Douro',
       'Guadia', 'Catala'],
      dtype='object', name='Catchment_name')


In [41]:
anova_df = df[
    df['Catchment_name'].isin(top8_names)
]

anova_df.head()

,Site_code,Latitude,Longitude,Country,Catchment_name,Galiza,Subsample,Calib_EFI_Medit,Calib_connect,Calib_hydrol,...,Squalius pyrenaicus,Squalius torgalensis,Thymallus thymallus,Tinca tinca,Zingel asper,Squalius sp,Barbatula sp,Phoxinus sp,Iberochondrostoma_sp,temp_ann_std
1,ES_02_0001,40.530188,-1.887796,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,-1.824026
2,ES_02_0002,40.595432,-1.928079,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,-1.455717
3,ES_02_0003,40.656184,-1.989831,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,-1.455717
4,ES_02_0004,40.676402,-2.036274,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,-1.363640
5,ES_02_0005,40.732830,-2.078003,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,-1.225524


In [42]:
# Create groups for ANOVA

groups = []

for catchment in top8_names:

    values = anova_df[
        anova_df['Catchment_name'] == catchment
    ]['Elevation_mean_catch'].dropna()

    groups.append(values)

# ANOVA
f_stat, p_value = stats.f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 227.9539828360387
P-value: 1.3695264820354472e-285


# Exercise 3 Interpretation

Null hypothesis (H0):
Mean elevation is equal across catchments.

Alternative hypothesis (H1):
At least one catchment has a different mean elevation.

If p-value < 0.05:
- reject H0
- conclude differences exist between catchments

In [43]:
# Kruskal-Wallis test

h_stat, p_value = stats.kruskal(*groups)

print("H-statistic:", h_stat)
print("P-value:", p_value)

H-statistic: 1335.3732750709976
P-value: 3.7056116510329714e-284


# Exercise 4 Interpretation

The Kruskal-Wallis test is the non-parametric equivalent of ANOVA.

Null hypothesis (H0):
The distributions of elevation are equal across catchments.

Alternative hypothesis (H1):
At least one catchment differs.

The test is useful when ANOVA assumptions are violated.